## 1. CARGA DE PARÁMETROS (Con valores por defecto para evitar errores)

In [0]:
import traceback
from pyspark.sql.functions import col, upper, trim, to_date, lit
from pyspark.sql.types import DecimalType, DateType

try:
    dbutils.widgets.text("catalog", "iowa_sales", "Catalog Name")
    dbutils.widgets.text("bronze_schema", "sales_bronze", "Bronze Schema Name")
    dbutils.widgets.text("silver_schema", "sales_silver", "Silver Schema Name")
    dbutils.widgets.text("bronze_table", "sales_raw", "Bronze Table Name")
    dbutils.widgets.text("silver_table", "sales_cleaned", "Silver Table Name")
    
    env = {
        "catalog": dbutils.widgets.get("catalog"),
        "sch_bronze": dbutils.widgets.get("bronze_schema"),
        "sch_silver": dbutils.widgets.get("silver_schema"),
        "tb_bronze": dbutils.widgets.get("bronze_table"),
        "tb_silver": dbutils.widgets.get("silver_table")
    }
    
    source_table = f"{env['catalog']}.{env['sch_bronze']}.{env['tb_bronze']}"
    target_table = f"{env['catalog']}.{env['sch_silver']}.{env['tb_silver']}"
    
    print(f"Configuración cargada. \nOrigen: {source_table} \nDestino: {target_table}")

except Exception as e:
    print("Error crítico al obtener los parámetros del notebook.")
    raise e

## 2. LECTURA, SEPARACIÓN (CUARENTENA) Y TRANSFORMACIÓN

In [0]:
try:
    print(f"Leyendo datos desde la capa Bronze...")
    df_sales_raw = spark.table(source_table)
    initial_row_count = df_sales_raw.count()
    
    # Columnas críticas que no pueden ser nulas
    critical_cols = [
        'invoice_line_no', 'date', "store", "category", "vendor_no", 
        "itemno", "pack", "bottle_volume_ml", "state_bottle_cost", 
        "state_bottle_retail", "sale_bottles"
    ]
    
    # Un registro va a cuarentena si está corrupto o si le falta un campo crítico
    quarantine_cond = (col("is_corrupted") == True)
    for c in critical_cols:
        quarantine_cond = quarantine_cond | col(c).isNull()
        
    # Separamos el flujo en dos DataFrames
    df_rejected = df_sales_raw.filter(quarantine_cond).withColumn("rejection_reason", lit("Corrupted or Missing Critical Data"))
    df_valid = df_sales_raw.filter(~quarantine_cond)
    # -------------------------------------------------------------------------

    print("Aplicando reglas de negocio, estandarización y deduplicación a registros válidos...")
    
    df_silver = (df_valid
        # 1. Deduplicar por llave primaria (sobre los registros ya filtrados)
        .dropDuplicates(["invoice_line_no"])
        
        # 2. Rellenar nulos para atributos descriptivos y el county_number
        .na.fill({
            "address": "UNKNOWN",
            "city": "UNKNOWN",
            "zipcode": "00000", 
            "county": "UNKNOWN",
            "category_name": "UNCATEGORIZED",
            "vendor_name": "UNKNOWN",
            "im_desc": "NO DESCRIPTION",
            "county_number": -1
        })
        
        # 3. Estandarización de Textos (Mayúsculas y sin espacios extra)
        .withColumn("name", upper(trim(col("name"))))
        .withColumn("address", upper(trim(col("address"))))
        .withColumn("city", upper(trim(col("city"))))
        .withColumn("county", upper(trim(col("county"))))
        .withColumn("category_name", upper(trim(col("category_name"))))
        .withColumn("vendor_name", upper(trim(col("vendor_name"))))
        .withColumn("im_desc", upper(trim(col("im_desc"))))
        
        # 4. Filtrar valores ilógicos (Precios o volúmenes en cero/negativos)
        # Nota: Los que fallen esta regla lógica matemática simplemente se descartan.
        .filter(
            (col("state_bottle_cost") > 0) & 
            (col("state_bottle_retail") > 0) &
            (col("bottle_volume_ml") > 0) &
            (col("sale_bottles") > 0)
        )
        
        # 5. Castear finanzas a Decimal y fechas a Date
        .withColumn("date", to_date(col("date")))
        .withColumn("state_bottle_cost", col("state_bottle_cost").cast(DecimalType(10, 2)))
        .withColumn("state_bottle_retail", col("state_bottle_retail").cast(DecimalType(10, 2)))
        
        # 6. Recalcular métricas
        .withColumn("sale_dollars", (col("state_bottle_retail") * col("sale_bottles")).cast(DecimalType(10, 2)))
        .withColumn("sale_liters", ((col("bottle_volume_ml") * col("sale_bottles")) / 1000).cast(DecimalType(10, 3)))
        .withColumn("sale_gallons", ((col("bottle_volume_ml") * col("sale_bottles")) / 3785.411784).cast(DecimalType(10, 3)))
        
        # 7. Renombrar y seleccionar
        .select(
            col("invoice_line_no"),
            col("date"),
            col("store").alias("store_number"),
            col("name").alias("store_name"),
            col("address"),
            col("city"),
            col("zipcode").alias("zip_code"),
            col("county_number"),
            col("county"),
            col("category").alias("category_code"),
            col("category_name"),
            col("vendor_no").alias("vendor_number"),
            col("vendor_name"),
            col("itemno").alias("item_number"),
            col("im_desc").alias("item_description"),
            col("pack"),
            col("bottle_volume_ml"),
            col("state_bottle_cost"),
            col("state_bottle_retail"),
            col("sale_bottles").alias("bottles_sold"),
            col("sale_dollars"),
            col("sale_liters").alias("volume_sold_liters"),
            col("sale_gallons").alias("volume_sold_gallons"),
            col("saved_date")
        )
    )
    
    final_row_count = df_silver.count()
    rejected_count = df_rejected.count()
    
    print(f"Conteo inicial (Bronze): {initial_row_count}")
    print(f"Enviados a Cuarentena: {rejected_count}")
    print(f"Conteo final (Silver): {final_row_count}")

except Exception as e:
    print("Error durante la transformación Silver.")
    raise e

## 3. ESCRITURA DE LA TABLA DE CUARENTENA (Auditoría)

In [0]:
try:
    quarantine_table = f"{env['catalog']}.{env['sch_silver']}.sales_quarantine"
    print(f"Guardando registros rechazados en {quarantine_table}...")
    
    df_rejected.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(quarantine_table)
        
except Exception as e:
    print("Error al escribir la tabla de cuarentena.")
    raise e

## 3. ESCRITURA EN CAPA SILVER

In [0]:
try:
    print(f"Escribiendo datos estandarizados en {target_table}...")
    
    df_silver.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(target_table)
        
    print("Escritura en capa Silver completada exitosamente.")

except Exception as e:
    print("Error al escribir en la tabla Silver.")
    raise e

dbutils.notebook.exit("Silver layer processing complete.")